In [1]:
''' Initial imports. Keep collapsed to reduce clutter. '''
import sys
import os
import matplotlib.pyplot as plt
%matplotlib widget
import numpy as np
import gvar as gv

from os import path
from IPython.display import clear_output
from copy import deepcopy
import gc
import importlib
import subprocess
import json
import ipywidgets as widgets
from ipywidgets import interact
from ipywidgets import interact_manual
from IPython.display import display

In [2]:
NNType = ["deuteron", "dineutron"]

DeuteronIrreps = [
    [("0", "T1g", 0)], [('0', 'T1g', 1)],
    [('1', 'A2', 0)], [('1', 'A2', 1)], 
    [('1', 'E', 0)], [('1', 'E', 1)], [('4', 'E', 0)], [('4', 'E', 1)],
    [('2', 'A2', 0)], [('4', 'A2', 0)], [('4', 'A2', 1)], 
    [('2', 'B1', 0)], [('2', 'B2', 0)], [('2', 'B2', 3)],
    [('3', 'A2', 0)], [('3', 'A2', 1)], [('3', 'E', 0)]
    ]

DineutronIrreps = [ [('Not yet or maybe never in this notebook')] ]

# ''' The lines below will define the bounds for the interactive widget. '''

# MinNStates = 0
# MaxNStates = 5

# GEVPMinTime = 0
# GEVPMaxTime = 12

# ''' Max & Min time for N and R fit range '''
# NMinTime = 0
# NMaxTime = 20
# RMinTime = 0
# RMaxTime = 15

# MinBlock = 0
# MaxBlock = 10 



In [3]:
''' Interactive widget layouts. Feel free to change where needed. Keep collapsed to reduce clutter. '''
layout = widgets.Layout(width='450px')
style={'description_width': '200px'}

# Fitting Procedure

## Define parameters

### Directories and Stuff

In [4]:
import nn_fit as fit
p = dict()
p["debug"]   = False
p["verbose"] = False
p["latex"]   = True
p["save"] = True
p["fitter"] = 'scipy_least_squares'
p["fpath"] = {"nucleon": "./data/cosmon_c103_r005-8_nucleon.hdf5", 
                "nn": "./data/cosmon_c103_r005-8_deuteron_Swave.hdf5"}
p['get_Zj']         = True
# if 'deuteron' in p["fpath"]["nn"]:
#     p['Zjn_values'] = f"result/deuteron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
# elif 'dineutron' in p["fpath"]["nn"]:
#     p['Zjn_values'] = f"result/dineutron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
p['show_Zjn']       = False
p['do_gevp']        = False #set to True if you want to do gevp if it was already done and saved

# Fitting

In [5]:
import timeit

In [6]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
def FitUpdateMain(Irrep, GEVP,TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio, Evp):
    p["masterkey"]  = [Irrep]
    p["t0"]         = GEVP[0]
    p["td"]         = GEVP[1]
    p['t_norm']     = TNorm
    p["ratio"]      = Ratio
    p["version"]    = AgCons
    p['gevp']       = Evp
    p["r_n_el"]     = NElastic
    p["block"]      = Block
    p['bs_seed']    = 'nn_c103_b%d' %p["block"]
    p["nstates"]    = NStates
    p["trange"]     = {"N": [NRange[0], NRange[1]], "R": [RRange[0], RRange[1]]}
    return p

def FitUpdateBootStrap(BootStrap,NbsMax,Nbs,NbsSub,Bs0Width,BSPriors):
    p["bootstrap"]  = BootStrap
    p['Nbs_max']    = NbsMax
    p["nbs"]        = Nbs
    p["nbs_sub"]    = NbsSub
    p['bs0_width']  = Bs0Width
    p['bs_prior']   = BSPriors
    return p 

def FitUpdateExtras(SvdStudy,SvdCut,Autotime,SigE0,SigEnn,PosZ,RatioType,IrrepType,GSConspire,RNInel,Ampi,AMN):
    p['svd_study']  = SvdStudy
    p['svdcut']     = SvdCut
    p["autotime"]   = Autotime
    p["sig_e0"]     = SigE0
    p["sig_enn"]    = SigEnn
    p["positive_z"] = PosZ
    p["ratio_type"] = RatioType
    p["gs_conspire"]= GSConspire
    p["r_n_inel"]   = RNInel
    p["ampi"]       = Ampi
    p["amn"]        = AMN
    p["irreps"]     = IrrepType
    p["dE_elastic"] = 2 * np.sqrt(p["amn"]**2 + 1 * (2 * np.pi / 48) ** 2) -2*p["amn"]
    return

def FitButton(b):
    if 'deuteron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"deuteron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    elif 'dineutron' in p["fpath"]["nn"]:
        p['Zjn_values'] = f"dineutron_Zjn_tNorm{p['t_norm']}_{p['gevp']}.h5"
    with FitButtonOut:
        clear_output()
        print('Full text output of fit can be toggled on and off in the Interactive Functions cell, FitButton function, and toggle the clear_output() line. ')
        print(f"Running fit: irrep={p['masterkey']} "
                f"t0={p['t0']} td={p['td']} "
                f"N={p['nstates']} block={p['block']} "
                f"trange={p["trange"]}"
                )
        %time fit.main(p)
        plt.close('all')

        # clear_output()

        print(f"Running fit: irrep={p['masterkey']} "
                f"t0={p['t0']} td={p['td']} "
                f"N={p['nstates']} block={p['block']} "
                f"trange={p["trange"]}"
                )
        print('Fitting completed.')
        print('Full text output of fit can be toggled on and off in the Interactive Functions cell, FitButton function, and toggle the clear_output() line. ')
    return 

In [7]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''

''' ----- Main Portion ----- '''
FitIrrep                = widgets.Dropdown(options=DeuteronIrreps, description='Fit Irrep: ', style=style, layout=layout)
FitGEVPt0td             = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
FitTNorm                = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
FitNElastic             = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
FitBlock                = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
FitNStates              = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
FitNRange               = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
FitRRange               = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
FitAgCons               = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)
FitEvpGevp              = widgets.Dropdown(options=['evp','gevp'], description='evp or gevp', orientation='horizontal', style=style, layout=layout)
FitRatio                = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

FitMainControls         = widgets.VBox([FitIrrep, FitGEVPt0td, FitTNorm, FitNStates, FitNElastic, FitNRange, FitRRange, FitBlock, FitAgCons, FitEvpGevp, FitRatio])

FitMainInteractive      = widgets.interactive_output(FitUpdateMain,
                                                            {   
                                                                "Irrep": FitIrrep,
                                                                "GEVP": FitGEVPt0td,
                                                                "TNorm": FitTNorm,
                                                                "NRange": FitNRange,
                                                                "RRange": FitRRange,
                                                                "NStates": FitNStates,
                                                                "AgCons": FitAgCons,
                                                                "NElastic": FitNElastic,
                                                                "Block": FitBlock,
                                                                "Ratio": FitRatio,
                                                                "Evp": FitEvpGevp
                                                            }
                                                        )

FitMainParamsAccordion      = widgets.Accordion(children=[widgets.VBox([FitMainControls, FitMainInteractive])])
FitMainParamsAccordion.set_title(0, "Main Fit Parameters")

''' ----- Boot Strap Portion ----- '''
FitBootStrap            = widgets.Checkbox(value=False, description='Bootstrap? ', style=style, layout=layout)
FitNbsMax               = widgets.IntText(value=5000,description='Max amount of BS samples', style=style, layout=layout)
FitNbs                  = widgets.IntText(value=5000,description='How many BS samples to do', style=style, layout=layout)
FitNbsSub               = widgets.IntText(value=100,description='How many BS samples to do before saving results', style=style, layout=layout)
BS0FitText              = widgets.HTML(value="During BS resampling, set the width of priors to 5*sigma where sigma is the gs uncertainty determined with boot0.  This is to stabilize BS resampling and help ensure the BS samples do not fall into a local minimum. ")
FitBS0Width             = widgets.IntText(value=5,description='bs0 width', style=style, layout=layout)
FitBSPrior              = widgets.Dropdown(options=['gs', 'all'], description='Randomize prior mean for gs or all priors', orientation='horizontal', style=style, layout=layout)

FitBootStrapControls    = widgets.VBox([BS0FitText, FitBootStrap,FitNbsMax,FitNbs,FitNbsSub,FitBS0Width,FitBSPrior])

FitBootStrapInteractive = widgets.interactive_output(FitUpdateBootStrap,
                                                            {   
                                                                "BootStrap": FitBootStrap, 
                                                                "NbsMax": FitNbsMax,
                                                                "Nbs": FitNbs,
                                                                "NbsSub": FitNbsSub,
                                                                "Bs0Width": FitBS0Width,
                                                                "BSPriors": FitBSPrior
                                                            }
                                                        )

FitBootStrapParamsAccordion     = widgets.Accordion(children=[widgets.VBox([FitBootStrapControls, FitBootStrapInteractive])])
FitBootStrapParamsAccordion.set_title(0, "Boot Strap Parameters")

''' ----- Extra Portion ----- ''' 
# Just the stuff that I don't know where to put for now. 
# If you would like, use the functions and widgets templates above to split this section up into what you find appropriate.
# Some of these are also not used anymore, so comment out or delete any that is unused to reduce clutter. 
# I have not found where irreps_ben is used except just existing in nn_parameters_base. 
FitSvdStudy             = widgets.Checkbox(value=False, description='Study SVD? ', style=style, layout=layout)
FitSvdCut               = widgets.FloatText(value=1e-8, description='SVD Cut', style=style, layout=layout)
FitAutotime             = widgets.IntSlider(value=10, min=0, max= 20, step=1, description="Time used to estimate mean gs energy prior", style=style, layout=layout)
FitSigE0                = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_gs", style=style, layout=layout)
FitSigEnn               = widgets.FloatSlider(value=1, description="Multiplication factor for meff[autotime] for prior width for deltaE_nn", style=style, layout=layout)
FitPosZ                 = widgets.Checkbox(value=True, description='Force overlaps to be positive? ', style=style, layout=layout)
FitRatioType            = widgets.Dropdown(options=['data'], description='Ratio Type', orientation='horizontal', style=style, layout=layout)
FitIrrepType            = widgets.Dropdown(options=["irreps_ben", "irreps"], description='Irreps type', orientation='horizontal', style=style, layout=layout)
FitGSConspire           = widgets.Checkbox(value=False, description='Only add deltaE for ground state? ', style=style, layout=layout)
FitRNInel               = widgets.IntSlider(value=2, min=0, max=10, step=1, description="R N Inelastic States", style=style, layout=layout)
FitAmpi                 = widgets.FloatSlider(value=0.310810, description="Ampi is used to construct excited state gaps", style=style, layout=layout)
FitAMN                  = widgets.FloatSlider(value=0.70262, description="Amn is used to estimate elastic excited state gaps", style=style, layout=layout)

# There is an extra dE_Elastic term that is in the function FitUpdateExtras, if you are looking for it. 

FitExtraControls        = widgets.VBox([FitSvdStudy,FitSvdCut,FitAutotime,FitSigE0,FitSigEnn,FitPosZ,FitRatioType,FitIrrepType,FitGSConspire,FitRNInel,FitAmpi,FitAMN])

FitExtraInteractive     = widgets.interactive_output(FitUpdateExtras,
                                                            {   
                                                                "SvdStudy": FitSvdStudy,
                                                                "SvdCut": FitSvdCut,
                                                                "Autotime": FitAutotime,
                                                                "SigE0": FitSigE0,
                                                                "SigEnn": FitSigEnn,
                                                                "PosZ": FitPosZ,
                                                                "RatioType": FitRatioType,
                                                                "IrrepType": FitIrrepType,
                                                                "GSConspire": FitGSConspire,
                                                                "RNInel": FitRNInel,
                                                                "Ampi": FitAmpi,
                                                                "AMN": FitAMN
                                                            }
                                                        )

FitExtraParamsAccordion = widgets.Accordion(children=[widgets.VBox([FitExtraControls, FitExtraInteractive])])
FitExtraParamsAccordion.set_title(0, "Extra Parameters")

''' ----- Button to Click ----- '''
FitButtonOut            = widgets.Output()

''' ----- Accordion stack ----- '''
FitAccordionStack       = widgets.VBox([FitMainParamsAccordion, FitBootStrapParamsAccordion, FitExtraParamsAccordion])

In [ ]:
RunFitButton           = widgets.Button(description='Run Fit')

RunFitButton.on_click(FitButton)

display(FitAccordionStack, RunFitButton, FitButtonOut)

Button(description='Run Fit', style=ButtonStyle())

Output()

# Plotting

In [9]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''
import plot_nn_stability_gevp as plotgevp

def Plot(filename,irrep,lims):
    argv = [
        "plot_nn_stability_gevp.py",
        filename
    ] 
    sys.argv = argv
    data = gv.load(filename)
    q_val = data[((irrep[0],), 'Q')]
    with PlotButtonOut:
        plt.close('all')
        plotgevp.main(params=irrep)
        clear_output()
        display(widgets.HTML(f"<b>Q: {q_val}</b>"))
        # inspect_all_figures()
        fig     = plt.gcf()
        TopAx   = fig.axes[0]
        MidAx   = fig.axes[1]
        if (lims[0]-lims[1])!=0:
            TopAx.set_ylim(lims[1],lims[0])
        if (lims[3]-lims[2])!=0:
            MidAx.set_ylim(lims[3],lims[2])
        plt.show()

def MakeFilename(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    nn_iso  = 'deuteron'
    irrep   = Irrep[0]
    RepPath = f"{irrep[0]}{irrep[1]}{irrep[2]}"
    ft0td   = f'{GEVP[0]}-{GEVP[1]}'
    fN      = f'{NRange[0]}-{NRange[1]}'
    fagcons = AgCons
    fnN     = NStates
    fe      = NElastic
    fR      = f'{RRange[0]}-{RRange[1]}'
    fRatio  = Ratio
    fBlock  = Block
    ftnorm  = TNorm
    filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}.pickle"
    return filename

def PlottingButton(b):
    with PlotButtonOut:
        clear_output()
        PlotFileName= MakeFilename(Irrep=PlotIrrep.value, GEVP=PlotGEVPt0td.value, 
                                   TNorm=PlotTNorm.value, NRange=PlotNRange.value, RRange=PlotRRange.value, 
                                   NStates=PlotNStates.value, AgCons=PlotAgCons.value, 
                                   NElastic=PlotNElastic.value, Block=PlotBlock.value, Ratio=PlotRatio.value)
        limits      = [Plot0UpLim.value,Plot0LowLim.value,Plot1UpLim.value,Plot1LowLim.value]
        
        Plot(filename=PlotFileName,irrep=PlotIrrep.value, lims=limits)

def PlotUpdate(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    return

In [10]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''
PlotIrrep                 = widgets.Dropdown(options=DeuteronIrreps, description='Choose Irrep: ', style=style, layout=layout)
PlotGEVPt0td              = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
PlotTNorm                 = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
PlotNElastic              = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
PlotBlock                 = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
PlotNStates               = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
PlotNRange                = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
PlotRRange                = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
PlotAgCons                = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)

PlotRatio                 = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

PlotControls              = widgets.VBox([PlotIrrep, PlotGEVPt0td, PlotTNorm, PlotNStates, PlotNElastic, PlotNRange, PlotRRange, PlotBlock, PlotAgCons, PlotRatio])

PlotInteractive           = widgets.interactive_output(PlotUpdate,
                                                            {   
                                                                "Irrep": PlotIrrep,
                                                                "GEVP": PlotGEVPt0td,
                                                                "TNorm": PlotTNorm,
                                                                "NRange": PlotNRange,
                                                                "RRange": PlotRRange,
                                                                "NStates": PlotNStates,
                                                                "AgCons": PlotAgCons,
                                                                "NElastic": PlotNElastic,
                                                                "Block": PlotBlock,
                                                                "Ratio": PlotRatio,
                                                            }
                                                        )

PlotParamsAccordion     = widgets.Accordion(children=[widgets.VBox([PlotControls, PlotInteractive])])
PlotParamsAccordion.set_title(0, "Plot Parameters")

Plot0UpLim              = widgets.FloatText(value=0, description='Top plot upper limit', style=style, layout=layout)
Plot0LowLim             = widgets.FloatText(value=0, description='Top plot lower limit', style=style, layout=layout)
Plot1UpLim              = widgets.FloatText(value=0, description='Middle plot upper limit', style=style, layout=layout)
Plot1LowLim             = widgets.FloatText(value=0, description='Middle plot lower limit', style=style, layout=layout)

PlotLimitsAccordion     = widgets.Accordion(children=[widgets.VBox([Plot0UpLim,Plot0LowLim,Plot1UpLim,Plot1LowLim])]) 
PlotLimitsAccordion.set_title(0, "Plot Limits")

PlotButtonOut = widgets.Output()

### Plot

In [11]:
PlotButton           = widgets.Button(description='Plot')

PlotButton.on_click(PlottingButton)

display(PlotParamsAccordion, PlotLimitsAccordion, PlotButton, PlotButtonOut)

Accordion(children=(VBox(children=(VBox(children=(Dropdown(description='Choose Irrep: ', layout=Layout(width='…

Accordion(children=(VBox(children=(FloatText(value=0.0, description='Top plot upper limit', layout=Layout(widt…

Button(description='Plot', style=ButtonStyle())

Output()

# Fit Parameters & Quality of Fit Metrics

### Plain Text

In [12]:
''' Interactive Functions. Feel free to change where needed. Keep collapsed to reduce clutter. '''

def PrintParameters(filename, irrep):

    data = gv.load(filename)

    IrrepData   = {k:v
                   for k,v in data.items()
                   if k[0][0] == irrep[0]}
    e_values    = {k:v
                   for k,v in IrrepData.items()
                   if  k[1].startswith('e')}
    z_values    = {k:v
                   for k,v in IrrepData.items()
                   if  k[1].startswith('z')}
    
    with PrintSingleNuke:
        clear_output()
        print(irrep)
        en_values   = {k:v
                       for k,v in e_values.items()
                       if k[0][1] == 'N'}
        zn_values   = {k:v
                       for k,v in z_values.items()
                       if k[0][1] == 'N'}
        for (ke,ve), (kz,vz) in zip(en_values.items(), zn_values.items()):
            print(f'{ke[1]}={ve}; p={ke[0][2]}')
            print(f'{kz[1]}={vz}; p={kz[0][2]}')
    with PrintDoubleNuke:
        clear_output()
        er_values   = {k:v
                       for k,v in e_values.items()
                       if k[0][1] == 'R'}
        zr_values   = {k:v
                       for k,v in z_values.items()
                       if k[0][1] == 'R'}
        for (ke,ve), (kz,vz) in zip(er_values.items(), zr_values.items()):
            print(f'{ke[1]}={ve}; p={ke[0][2]}')
            print(f'{kz[1]}={vz}; p={kz[0][2]}')
    with PrintQuality:
        clear_output()
        quality_vals= {k:v
                   for k,v in IrrepData.items()
                   if len(k[0]) == 1}
        q_val       = quality_vals[(irrep[0],),'Q']
        chi2dof     = quality_vals[(irrep[0],),'chi2']/quality_vals[(irrep[0],),'dof']
        loggbf      = quality_vals[(irrep[0],),'logGBF']
        print(f'Q: {q_val}')
        print(f'chi2/dof: {chi2dof}')
        print(f'logGBF: {loggbf}')
    with PrintSettings:
        clear_output()
        print(f'svdcut/svdn: {IrrepData[(irrep[0],),'svdcut']}/{IrrepData[(irrep[0],),'svdn']}')
        print('Not quite sure where this egs term goes')
        print(f'egs: {data[irrep[0],'egs']}')
    display(PrintParamsAccordion)

def MakeFilename(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    nn_iso  = 'deuteron'
    irrep   = Irrep[0]
    RepPath = f"{irrep[0]}{irrep[1]}{irrep[2]}"
    ft0td   = f'{GEVP[0]}-{GEVP[1]}'
    fN      = f'{NRange[0]}-{NRange[1]}'
    fagcons = AgCons
    fnN     = NStates
    fe      = NElastic
    fR      = f'{RRange[0]}-{RRange[1]}'
    fRatio  = Ratio
    fBlock  = Block
    ftnorm  = TNorm
    # filename = f"result/NN_deuteron_tnorm3_t0-td_4-8_N_n3_t_3-20_NN_conspire_e0_t_3-15_ratio_False_block2.pickle"
    filename = f"result/{RepPath}/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fBlock}.pickle"
    return filename

def PrintButton(b):
    with PrintButtonOut:
        clear_output()
        FitFileName = MakeFilename(Irrep=ChooseIrrep.value, GEVP=ChooseGEVPt0td.value, 
                                   TNorm=ChooseTNorm.value, NRange=ChooseNRange.value, RRange=ChooseRRange.value, 
                                   NStates=ChooseNStates.value, AgCons=ChooseAgCons.value, 
                                   NElastic=ChooseNElastic.value, Block=ChooseBlock.value, Ratio=ChooseRatio.value)
        PrintParameters(filename=FitFileName,irrep=ChooseIrrep.value)

def update(Irrep, GEVP, TNorm, NRange, RRange, NStates, AgCons, NElastic, Block, Ratio):
    return

In [13]:
''' Interactive Widgets. Feel free to change where needed. Keep collapsed to reduce clutter. '''
ChooseIrrep                 = widgets.Dropdown(options=DeuteronIrreps, description='Choose Irrep: ', style=style, layout=layout)
ChooseGEVPt0td              = widgets.IntRangeSlider(value=[3,8], min=0, max=12, step=1, description='GEVP t0-td: ', style=style, layout=layout)
ChooseTNorm                 = widgets.IntSlider(value=3, min=0, max=5, step=1, description='tNorm: ', style=style, layout=layout)
ChooseNElastic              = widgets.IntSlider(value=0, min=0, max=5, step=1, description='R N Elastic states: ', style=style, layout=layout)
ChooseBlock                 = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Block: ', style=style, layout=layout)
ChooseNStates               = widgets.IntSlider(value=3, min=0, max=5, step=1, description='Nstates: ', style=style, layout=layout)
ChooseNRange                = widgets.IntRangeSlider(value=[3,20], min=0, max=20, step=1, description='N Range t0-td: ', style=style, layout=layout)
ChooseRRange                = widgets.IntRangeSlider(value=[3,15], min=0, max=15, step=1, description='R Range t0-td: ', style=style, layout=layout)
ChooseAgCons                = widgets.Dropdown(options=['conspire', 'agnostic'], description='Agnostic or Conspiracy? ', orientation='horizontal', style=style, layout=layout)

ChooseRatio                 = widgets.Checkbox(value=False, description='Ratio? ', style=style, layout=layout)

controls                    = widgets.VBox([ChooseIrrep, ChooseGEVPt0td, ChooseTNorm, ChooseNStates, ChooseNElastic, ChooseNRange, ChooseRRange, ChooseBlock, ChooseAgCons, ChooseRatio])

PrintInteractive            = widgets.interactive_output(update,
                                                            {   
                                                                "Irrep": ChooseIrrep,
                                                                "GEVP": ChooseGEVPt0td,
                                                                "TNorm": ChooseTNorm,
                                                                "NRange": ChooseNRange,
                                                                "RRange": ChooseRRange,
                                                                "NStates": ChooseNStates,
                                                                "AgCons": ChooseAgCons,
                                                                "NElastic": ChooseNElastic,
                                                                "Block": ChooseBlock,
                                                                "Ratio": ChooseRatio,
                                                            }
                                                        )

ChoosePrintParamsAccordion  = widgets.Accordion(children=[widgets.VBox([controls, PrintInteractive])])
ChoosePrintParamsAccordion.set_title(0, "Fit Parameters")

''' Print Parameters in 4 Accordions '''

PrintSingleNuke             = widgets.Output()
PrintDoubleNuke             = widgets.Output()
PrintQuality                = widgets.Output()
PrintSettings               = widgets.Output()

PrintParamsAccordion        = widgets.Accordion(children=[PrintSingleNuke, PrintDoubleNuke, PrintQuality, PrintSettings])
PrintParamsAccordion.set_title(0, "Single Nucleon Parameters")
PrintParamsAccordion.set_title(1, "Double Nucleon Parameters")
PrintParamsAccordion.set_title(2, "Quality of Fit Metrics")
PrintParamsAccordion.set_title(3, "Settings")

PrintButtonOut              = widgets.Output()

In [14]:
''' Actual Parameter Printing '''
PrintParamsButton           = widgets.Button(description='Display Plain Text')

PrintParamsButton.on_click(PrintButton)

display(ChoosePrintParamsAccordion, PrintParamsButton, PrintButtonOut)

Accordion(children=(VBox(children=(VBox(children=(Dropdown(description='Choose Irrep: ', layout=Layout(width='…

Button(description='Display Plain Text', style=ButtonStyle())

Output()

### Plots

# Old Functional Code Kept Around In Case It Is Needed Again

## Functional since v0.1.2

In [15]:
run012cells = False

### Looping Parameters

The parameters here can and will be looped over. This block and the next covers all the bash script functionality. 

pt0 and ptd needs to be the same length. The gevp times will be t0-td of the element in the same spots.

In [16]:
if run012cells:
    ''' Feeds p["t0"] '''
    pt0     = [4]
    ''' Feeds p["td"] '''
    ptd     = [8]
    ''' Feeds p["nstates"] '''
    pnN     = [3]
    ''' Feeds p["block"] '''
    pblock  = [8]
    ''' Feeds p["trange"] N part. Number before 20 '''
    pN      = [3]
    ''' Feeds p["trange"] R part. Number before 15 '''
    pR      = [3]

In [17]:
''' Memory leak fixed fitting procedure for looping over large amounts of fits '''
if run012cells:
    for i in range(len(pt0)):
        for j in range(len(pblock)):
            for k in range(len(pnN)): 
                for l in range(len(pN)):
                    for m in range(len(pR)):

                        p["t0"]     = pt0[i]
                        p["td"]     = ptd[i]
                        p["block"]  = pblock[j] 
                        p['bs_seed']   = 'nn_c103_b%d' %p["block"]
                        p["nstates"]    = pnN[k]
                        p["trange"]      = {"N": [pN[l], 20], "R": [pR[m], 15]}
                        print(
                                f"Running fit: t0={p['t0']} td={p['td']} "
                                f"N={p['nstates']} block={p['block']} "
                                f"trange=[{pN[l]},20] [{pR[m]},15]"
                            )

                        gv.switch_gvar()

                        params_json = json.dumps(p)

                        result = subprocess.run(
                            [sys.executable, "-u", "run_fit.py", params_json],
                            capture_output=True,
                            text=True
                        )

                        print("STDOUT:\n", result.stdout)
                        print("STDERR:\n", result.stderr)

                        if result.returncode != 0:
                            raise RuntimeError("Fit crashed")
                        
                        gv.restore_gvar()

                        plt.close("all")
                        clear_output()
                        gc.collect()
                        

    print("Fitting Completed")

### Main Plotting Parameters

In [18]:
if run012cells:
    ''' v0.1.2: 
    Currently only supports plotting 1 or all.
    Put in 1 irrep to plot just 1 irrep. Toggle PlotAll to True to plot every irrep (potentially takes minutes). 
    '''

    ''' Deuteron irreps:
        [("0", "T1g", 0)], [('0', 'T1g', 1)],
        [('1', 'A2', 0)], [('1', 'A2', 1)], 
        [('1', 'E', 0)], [('1', 'E', 1)], [('4', 'E', 0)], [('4', 'E', 1)],
        [('2', 'A2', 0)], [('4', 'A2', 0)], [('4', 'A2', 1)], 
        [('2', 'B1', 0)], [('2', 'B2', 0)], [('2', 'B2', 3)],
        [('3', 'A2', 0)], [('3', 'A2', 1)], [('3', 'E', 0)]     '''

    irrep = [("0", "T1g", 0)]    # Copy & paste with brackets. Should look like [(irrep)]. 

    ''' Plot every irrep. 
        v0.1.2: Overrides the single plot due to UseFitParams, and the irrep selection above. 
    '''
    PlotAll         = False # True False

    ''' Clear text output for plotting cell '''
    PlotClearText   = True # True False

    ''' ------ Auto-fill flags ------ '''
    ''' Set to true to auto-fill all relevant parameters for plotting. '''

    ''' Auto-fill all "fit file directory" and "additional" parameters using the fitting parameters. '''
    ''' v0.1.2: If you decided to fit everything. It will default to ("0", "T1g", 0). 
                It displays the 1st fit done. That is the 1st element in the "looping parameters". 
    '''
    UseFitParams    = True # True False

    ''' Auto-fill all "additional" plotting parameters with defaults. Will overwrite "additional" parts of UseFitParams. '''
    UseDefaults     = True # True False

### Fit File Directory Parameters
You can skip doing this and go to plot cell -- cell with plot in text above it -- and copy paste the file name into there.

In [19]:
if run012cells:
    ''' System. 'dineutron' or 'deuteron' '''
    nn_iso  = 'deuteron'

    ''' gevp time. In the form 't0-td' '''
    ft0td   = '4-8'

    ''' 'conspire' or 'agnostic' '''
    fagcons = 'conspire'

    ''' t_norm. In the form 'int' '''
    ftnorm  = 3

    ''' block '''
    fblock  = 8

    ''' N part of t_range. The number before 20. '''
    fN      = 4

    ''' R part of t_range. The number before 15. '''
    fR      = 2

    ''' Ratio '''
    fRatio  = False

    ''' r_n_el '''
    fe      = 0

    ''' nstates '''
    fnN     = 3

### Additional Plotting Parameters

In [20]:
if run012cells:
    ''' Number of exponentials in single nucleon to sweep over. Default: [3] '''
    n_N     = [2,3,4]   
    ''' Number of elastic e.s. to try.                          Default: [0] '''
    nn_el   = [0]    
    ''' List of GEVP times in t0-td format                      Default: ['4-8', '4-10', '5-10','5-12', '6-10', '6-12'] '''
    gevp    = ['4-8', '4-10', '5-10','5-12', '6-10', '6-12']      
    ''' Values of t_min in NN fit.                              Default: range(2,9) '''
    tmin    = range(2,9)

    ''' Fit from RATIO correlator?                              Default: False '''
    ratio   = False
    ''' Load evp (vs gevp) data?                                Default: True '''
    evp     = True
    ''' Use gs only conspiracy model?                           Default: False '''
    gs_cons = False

    ''' The 3 below are not covered by UseFitParams'''

    ''' What type of figure?    Types: "pdf" , "png"            Default: "pdf" '''
    fig_type= "pdf"

    ''' If test==True, only do T1g                              Default: False '''
    test    = False
    ''' Add extra debug print statements?                       Default: False '''
    debug   = False

### Auto-fill Cell

In [21]:
if run012cells:
    if UseFitParams:
        irrep   = p["masterkey"][0]

        ''' File Directory'''
        if 'deuteron' in p["fpath"]["nn"]:
            nn_iso  = 'deuteron'
        elif 'dineutron' in p["fpath"]["nn"]:
            nn_iso  = 'dineutron'
        ft0td   = f"{p["t0"]}-{p["td"]}"
        fagcons = deepcopy(p["version"])
        ftnorm  = deepcopy(p["t_norm"])
        fblock  = f"{p["block"]}"
        fN      = f"{p['trange']['N'][0]}-{p['trange']['N'][1]}"
        fR      = f"{p['trange']['R'][0]}-{p['trange']['R'][1]}"
        fRatio  = deepcopy(p["ratio"])
        fe      = deepcopy(p["r_n_el"])
        fnN     = f"{p["nstates"]}"

        ''' Additional Parameters'''
        # n_N     = deepcopy(pnN)
            
        # nn_el   = [deepcopy(p["r_n_el"])]

        # gevp    = []
        # for t0, td in zip(pt0, ptd):
        #     gevp.append(f"{t0}-{td}")

        # tmin    = deepcopy(pN)

        # ratio   = deepcopy(p["ratio"])

        # if p["gevp"] == "evp":
        #     evp = True
        # else: evp = False

        # gs_cons = deepcopy(p["gs_conspire"])

    if PlotAll:
        irrep = []

    if UseDefaults:
        for var in ["n_N","nn_el","ratio","gevp","evp","tmin","gs_cons","fig_type","test","debug"]:
            if var in locals():
                del locals()[var]

In [22]:
if run012cells:

    import plot_nn_stability_gevp as plotgevp

    filename = f"result/NN_{nn_iso}_tnorm{ftnorm}_t0-td_{ft0td}_N_n{fnN}_t_{fN}_NN_{fagcons}_e{fe}_t_{fR}_ratio_{fRatio}_block{fblock}.pickle"
    argv = [
        "plot_nn_stability_gevp.py",
        filename
    ] 

    # argv = [
    #     "plot_nn_stability_gevp.py",
    #     "result/NN_deuteron_tnorm3_t0-td_4-8_N_n3_t_4-20_NN_conspire_e0_t_2-15_ratio_False_block8.pickle"
    # ]   # Comment out the 1st argv and uncomment this argv. 
        # Copy & paste the file name after result/ into space above to circumvent the "fit file parameter" portion.  

    if 'n_N' in locals():
        argv += ["--n_N"] + [str(x) for x in n_N]

    if 'nn_el' in locals():
        argv += ["--nn_el"] + [str(x) for x in nn_el]

    if 'gevp' in locals():
        argv += ["--gevp"] + [str(x) for x in gevp]

    if 'tmin' in locals():
        argv += ["--tmin"] + [str(x) for x in tmin]

    if 'fig_type' in locals():
        argv += ["--fig_type", fig_type]

    if 'ratio' in locals():
        if ratio:
            argv.append("--ratio")

    if 'evp' in locals():
        if not evp:
            argv.append("--evp")

    if 'gs_cons' in locals():
        if gs_cons:
            argv.append("--gs_cons")

    if 'test' in locals():
        if test:
            argv.append("--test")

    if 'debug' in locals():
        if debug:
            argv.append("--debug")

    sys.argv = argv

    plotgevp.main(params=irrep)

    if PlotClearText:
        clear_output()

    plt.show()

### Summary Plot

In [23]:
if run012cells:
    argv.append("--summary")

    allirreps = []

    plotgevp.main(params=allirreps)

    if PlotClearText:
        clear_output()

    plt.show()